# An introductory example

## Data cleaning with "pandas"

#### A quick overview of some simple but useful python functionality that is covered in the course content

The example specifically showcases some of python's usefulness in opening, handling, viewing and plotting data, using as an example various ocean depth profiles with different formatting and variables. 

It is not necessary (or expected) for this to be understood yet -- the idea at this stage is to give some context to the course content coming up, and demonstrate how a lot can be achieved using short and readable commands. I have added comments for completeness, but these concepts will make more sense in the individual lessions. In addition to the data handling described above, the example also relies on various commonly-used python features that will be covered in the course, such as: 

- The use of python packages, and their complementarity to each other
- The object-oriented nature of python
- the use of text strings and their seamless use in indexing
- Using dictionaries to easily facilitate inputs and outputs   

### Load in python packages

These each handle different things, such as reading and handling data (pandas; numpy; xarray), and plotting data (matplotlib; cartopy). These open-source packages are often developed to complement each other.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Open some csv data (excel spreadsheet) of a vertical profile of ocean variables

Read a csv dataset using the pandas package. To do this we first call the package we want to use (in this case pandas, shortened to pd for quicker typing), then use its read_csv functionality for reading the datafile (specified after the full stop. i.e. pd.read_csv('/text/string/to/file/location'). 

In [2]:
df_dataset1 = pd.read_csv('../data/CTD_25pt3N_42pt3W.csv')
df_dataset1.head(15)

,Date,Profile_ID,Depth_m,Temp_K,Sal_psu,Oxygen_umolkg,Nitrate_umolkg
0,2025-01-15,CTD_1,0,300.897503,35.061530,219.744474,-2.075705
1,2025-01-15,CTD_1,10,300.658480,35.118627,216.061637,-0.778733
2,2025-01-15,CTD_1,20,300.347929,35.170291,212.583226,0.448151
3,2025-01-15,CTD_1,30,299.947447,35.217039,209.298195,1.608733
4,2025-01-15,CTD_1,50,298.790578,35.297611,203.267033,3.745132
5,2025-01-15,CTD_1,75,296.453550,35.378110,196.641318,6.102520
6,2025-01-15,CTD_1,100,292.989674,35.440802,190.907213,8.154213
7,2025-01-15,CTD_1,150,284.801927,35.527651,181.666349,11.493938
8,2025-01-15,CTD_1,200,279.866961,35.580328,174.787983,14.023663
9,2025-01-15,CTD_1,300,277.790817,35.631657,165.965766,17.391280


pandas has a lot of functionality, including methods for data manipulation and for directly plotting the data. 

### Identify and plot a variable using its name 

Select the Temp_K variable, showing temperature in Kelvin for that particular profile. Notice that it handles text easily, meaning we can directly use the name of the variable rather than numbered index locations. 

In [ ]:
df_dataset1.plot(x="Temp_K", y="Depth_m",ylim=[1100,0],marker='.',linewidth=0,legend=False)

You can do something a little fancier to calcualte the mean and standard deviation over these profiles. Uses a function called 'groupby' to group data together with matching depth. 

In [ ]:
variable = 'Temp_K'
coordinate = 'Depth_m'

stats = df_dataset1.groupby(coordinate)[variable].agg(["mean", "std"])
plt.plot(stats['mean'], stats.index, marker="o", label='mean '+variable)
plt.fill_betweenx(stats.index,stats["mean"] - stats["std"],stats["mean"] + stats["std"],alpha=0.3,label='Stdev '+variable)
plt.legend()
plt.ylim([1100,0])
plt.ylabel('Depth (m)')
plt.xlabel('Temp (°C)')
plt.title('Mean & Stdev of Temp')
plt.grid()

## Handling multiple datasets with different formatting

Let's read in another datafile from a different source. Note that it has different formats, names, units, coordinates, resolution, ordering etc. compared to the first

In [ ]:
df_dataset2 = pd.read_csv('../data/ARGO_LabSea.csv')
df_dataset2.head(15)

This second data format makes it difficult to directly compare the datasets without modifying them. A good way to do this is with a dictionary that remaps variable names onto a standard set of names (so called key-value pairing), and can identify any unit conversions etc. This then allows you to work with the original data without manually modifying it. 

In [ ]:
### Set the relevant meta data for each data file

inputs = {
    
    "LabSea_ARGO": {
        "filename": "../data/ARGO_LabSea.csv",
        "depth": "pressure_depth", 
        "time" : "sample_time",
        "columns": {
                    "temperature": "temperature_C", 
                    "salinity": "salinity_gkg",
                    "nitrate" : "nitrate_concentration",
                    "po4" : "phosphate_concentration",
                   },
        "temp_offset_to_celsius": 0.0
    },
    
    "Subtropical_CTD": {
        "filename": "../data/CTD_25pt3N_42pt3W.csv",
        "depth": "Depth_m", 
        "time" : "Date",
        "columns": {
                    "temperature": "Temp_K", 
                    "salinity": "Sal_psu",
                    "nitrate" : "Nitrate_umolkg"
                    },
        "temp_offset_to_celsius": -273.15
    },
    
    "IrmSea_ARGO": {
        "filename": "../data/ARGO_IrmingerSea.csv",
        "depth": "pressure_depth", 
        "time" : "sample_time",
        "columns": {
                    "temperature": "temperature_C", 
                    "salinity": "salinity_gkg",
                    "nitrate" : "nitrate_concentration",
                    "po4" : "phosphate_concentration",
                   },
        "temp_offset_to_celsius": 0.0
    },
        
}

Now plot the mean and standard deviation from all datasets. It just loops over the listed datasets and the listed 'columns' from the dictinary, then plots in essentially the same way as above. 

In [ ]:
fig, axes = plt.subplots(1, len(inputs["LabSea_ARGO"]["columns"]), figsize=(4*len(inputs["LabSea_ARGO"]["columns"]), 5), sharey=True)
output_profiles = {}   # Initialise the output dictionary

for profile in inputs.keys():
    output_profiles[profile] = {}
    df = pd.read_csv(inputs[profile]["filename"])
    cols = inputs[profile]["columns"]
    for num, variable in enumerate(inputs[profile]["columns"].keys()):   
        
        ### identify data and variable
        offset = inputs[profile]["temp_offset_to_celsius"] if variable=='temperature' else 0
        depth = df[inputs[profile]["depth"]]
        stats = df.groupby(inputs[profile]['depth'])[cols[variable]].agg(["mean", "std"])

        # Plot the data
        axes[num].plot(stats['mean']+offset, stats.index, marker="o", label=profile)
        axes[num].fill_betweenx(stats.index,stats["mean"]+offset - stats["std"],stats["mean"]+offset + stats["std"],alpha=0.3)
        axes[num].legend()
        axes[num].set_ylim([2200,0])
        axes[num].set_xlabel(variable)
        axes[num].set_title('Mean & stdev of '+variable)
        axes[num].grid(True)

To add another variable from the datafile, just add it to the 'columns'. If you want to add another datafile, add another code block to the dictionary. 

You can also then store the output data in an automated output dictionary that matches the format of the input dictionary

In [ ]:
# Store the processed outputs in a matching output dictionary in an automated way 

for profile in inputs.keys():
    output_profiles[profile] = {}
    df = pd.read_csv(inputs[profile]["filename"])
    cols = inputs[profile]["columns"]
    for num, variable in enumerate(inputs[profile]["columns"].keys()):   
        
        ### identify data and variable
        offset = inputs[profile]["temp_offset_to_celsius"] if variable=='temperature' else 0
        depth = df[inputs[profile]["depth"]]
        stats = df.groupby(inputs[profile]['depth'])[cols[variable]].agg(["mean", "std"])

        # Store the data in a matching dictionary
        output_profiles[profile]["depth_m"] = df[inputs[profile]["depth"]]
        output_profiles[profile]["filename"] = inputs[profile]["filename"]
        output_profiles[profile][variable] = {'mean' : stats['mean']+offset, 'stdev' : stats["std"]}

## Some useful things this script demonstrates
- Recognition of text strings for indexing -- avoids pre-processing data to get the desidered order
- Use of dictionaries -- allows all metadata to be identified and standardised in one place, then used to create an output dictionary with matching formatting for easy access of processed data.
- Gives an introduction to the pandas package for handling data tables